# 03 - Table 5 Ablation Study

Chạy ablation độc lập trên Colab. Biến thể NoFeatureSelect dùng toàn bộ features đã preprocess từ Notebook 01.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_PATH = '/content/drive/MyDrive/Nhom28_CyberDetect_MLP_Final'
os.makedirs(PROJECT_PATH, exist_ok=True)
%cd {PROJECT_PATH}
print('PROJECT_PATH =', PROJECT_PATH)


In [ ]:
!pip -q install pandas numpy tensorflow scikit-learn pyarrow matplotlib


In [ ]:

import os, time, math, random
import numpy as np, pandas as pd, tensorflow as tf, matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Dense, BatchNormalization, Dropout
from tensorflow.keras.callbacks import EarlyStopping, LearningRateScheduler
from tensorflow.keras.optimizers import Adam
OUT=f'{PROJECT_PATH}/data/colab_processed'; RESULTS=f'{PROJECT_PATH}/results'; os.makedirs(RESULTS,exist_ok=True)
y_train=pd.read_parquet(f'{OUT}/y_train.parquet')['label'].values.astype(int); y_test=pd.read_parquet(f'{OUT}/y_test.parquet')['label'].values.astype(int)
X_top_tr=pd.read_parquet(f'{OUT}/X_train_top30.parquet').values.astype('float32'); X_top_te=pd.read_parquet(f'{OUT}/X_test_top30.parquet').values.astype('float32')
X_all_tr=pd.read_parquet(f'{OUT}/X_train_all.parquet').values.astype('float32'); X_all_te=pd.read_parquet(f'{OUT}/X_test_all.parquet').values.astype('float32')
def seed(): random.seed(42); np.random.seed(42); tf.random.set_seed(42)
def cosine(e,T=100): return 1e-5 + .5*(1e-3-1e-5)*(1+math.cos(math.pi*e/T))
def build(dim,variant):
    m=Sequential([Input(shape=(dim,))])
    for u in [512,256,128]:
        m.add(Dense(u,activation='relu'))
        if variant!='no_batchnorm': m.add(BatchNormalization())
        if variant!='no_dropout': m.add(Dropout(0.3))
    m.add(Dense(1,activation='sigmoid')); m.compile(Adam(1e-3),'binary_crossentropy',metrics=['accuracy']); return m
def run(name,variant,Xtr,Xte,scheduler=True):
    seed(); model=build(Xtr.shape[1],variant); cb=[EarlyStopping(monitor='val_loss',patience=10,restore_best_weights=True)]
    if scheduler: cb.append(LearningRateScheduler(lambda e: cosine(e)))
    w=compute_class_weight('balanced',classes=np.unique(y_train),y=y_train); cw=dict(enumerate(w)); t=time.time(); h=model.fit(Xtr,y_train,validation_split=.2,epochs=100,batch_size=64,callbacks=cb,class_weight=cw,verbose=2)
    prob=model.predict(Xte,verbose=0).reshape(-1); yp=(prob>=.5).astype(int)
    return {'variant':name,'accuracy':accuracy_score(y_test,yp),'precision':precision_score(y_test,yp,zero_division=0),'recall':recall_score(y_test,yp,zero_division=0),'f1':f1_score(y_test,yp,zero_division=0),'roc_auc':roc_auc_score(y_test,prob),'train_time_sec':time.time()-t,'epochs_ran':len(h.history['loss'])}
rows=[run('Full Model','full',X_top_tr,X_top_te),run('MLP-NoFeatureSelect','full',X_all_tr,X_all_te),run('MLP-NoBatchNorm','no_batchnorm',X_top_tr,X_top_te),run('MLP-NoDropout','no_dropout',X_top_tr,X_top_te),run('MLP-NoScheduler','full',X_top_tr,X_top_te,False)]
df=pd.DataFrame(rows); df.to_csv(f'{RESULTS}/colab_ablation_metrics.csv',index=False); display(df)
df.set_index('variant')[['accuracy','precision','recall','f1','roc_auc']].plot(kind='bar',figsize=(11,5)); plt.ylim(0,1.05); plt.title('Table 5 Ablation Study'); plt.xticks(rotation=20,ha='right'); plt.tight_layout(); plt.savefig(f'{RESULTS}/fig_colab_ablation.png',dpi=150); plt.show()
